In [2]:
import json
import sys
import numpy as np
sys.path.append(r"D:\Desktop_fake\MawileBot\home\MawileBot\src")
from poke_lib import get_poke_bst, similar_pokemon_name, next_gym, generate_all_types_combo, poke_cell_gym 
from BeatlesBoy_utils import find_evo_at_level_x,load_x_from_json, calculate_bonus_via_types, load_team_from_json_simple
import random

ModuleNotFoundError: No module named 'poke_lib'

## Da lanciare in D:\Desktop_fake\MawileBot (NON D:\Desktop_fake\MawileBot\home\MawileBot )

In [ ]:
async def pokemon_utility_old(pokemon,lvl):
    if pokemon is  None:
        return 0
    #TODO: implement a better utility function
        
    pokemon = await similar_pokemon_name(pokemon.lower(), r = False)
    fully_evo = await find_evo_at_level_x(pokemon, 100)
    max_bst = await get_poke_bst(fully_evo)
    if fully_evo in (await load_x_from_json("mega")):
        max_bst = await get_poke_bst(fully_evo+'-mega')
        print(f"BST di Mega {await find_evo_at_level_x(pokemon, 100)} a livello 100: {max_bst}")

    else:
        print(f"BST di {await find_evo_at_level_x(pokemon, 100)} a livello 100: {max_bst}")

    utility_bst = max_bst/620*10
    utility_lvl = lvl/10

    utility = utility_bst * 0.7 + utility_lvl * 0.3
    
    return round(utility, 2)  # 0-10 scale


In [ ]:
async def pokemon_utility(pokemon,lvl, catch = False):
    if pokemon is  None:
        return 0
    #TODO: implement a better utility function
        
    pokemon = await similar_pokemon_name(pokemon.lower(), r = False)
    fully_evo = await find_evo_at_level_x(pokemon, 100)
    max_bst = await get_poke_bst(fully_evo)
    if fully_evo in (await load_x_from_json("mega")):
        max_bst = await get_poke_bst(fully_evo+'-mega')
        print(f"BST di Mega {await find_evo_at_level_x(pokemon, 100)} a livello 100: {max_bst}")

    else:
        print(f"BST di {await find_evo_at_level_x(pokemon, 100)} a livello 100: {max_bst}")


    utility_bst = max_bst/620*10
    utility_lvl = lvl/10

    utility = utility_bst * 0.7 + utility_lvl * 0.3

    try:
        utility += await next_gym_bonus(pokemon, lvl, catch)
    except Exception as e:
        print(f"Error calculating next gym bonus: {e}")
        
    return round(utility, 2)  # 0-10 scale (except bonus next gym)

In [ ]:
import pypokedex as poke

gym_type = 'rock'
multiplier = 20
low_power = 30
power  = 20
pokemon = 'pikachu'

def win_perc_over_gym(gym_type, low_power, pokemon, power, multiplier):
    all_types_combo = generate_all_types_combo(gym_type)
    wins = 0
    for t in all_types_combo:
        types2 = poke.get(name=pokemon).types
        bonus = calculate_bonus_via_types(t, types2 ,multiplier)
        # print(f"Types: {t}, Bonus: {bonus}, Power: {power}, Low Power: {low_power}")
        if power - bonus[0] + bonus[1] > low_power:
            wins += 1
    return wins / len(all_types_combo)

async def next_gym_bonus(pokemon, lvl, catch = False):

    gym_type,multiplier,casella_gym = (await next_gym())
    _, _, enemy_powers, multiplier, _ = poke_cell_gym(casella_gym)
    low_power = min(enemy_powers)

    # if the average win percentage of the current team is already above 66%, we can skip the bonus calculation for a new catch
    if catch == True:
        try: 
            team = (await load_team_from_json_simple())
        except:
            team = [["chespin",7],["unown",5],["pancham",5],
                    ["sableye",5],["skorupi",5]]
        w_p = []
        for (p, l) in team:
            if p is not None:
                pp = round(await get_poke_bst(p)*(l+3)/100)
                w_p.append(win_perc_over_gym(gym_type, low_power, p, pp, multiplier))

        w_p = sorted(w_p, reverse=True)[:6]
        w_p += [0] * (6 - len(w_p))
        avg_win_perc = sum(w_p) / len(w_p)
        if avg_win_perc > 0.66:
            return 0

    # IF NOT, we need a new pokemn ASAP!
     
    power = round(await get_poke_bst(pokemon)*lvl/100)
    win_perc = win_perc_over_gym(gym_type, low_power, pokemon, power, multiplier)
    print(f"Win percentage against next gym: {win_perc*100:.2f}%")

    power_plus5 = round(await get_poke_bst(pokemon)*(lvl+5)/100)
    win_perc_plus5 = win_perc_over_gym(gym_type, low_power, pokemon, power_plus5, multiplier)
    print(f"Win percentage against next gym: {win_perc_plus5*100:.2f}%")

    if win_perc_plus5 > 0.75:
        if win_perc < 0.75:
            return 5  # This will also boost in training, not only in catches!
        
    if win_perc > 0.75 and catch == True:
        return 999  # We need to catch this beast!
    
    return 0

In [ ]:
print(await pokemon_utility('Pikachu',7))
print(await pokemon_utility('Pikachu',7, catch=True))
print('\n')
print(await pokemon_utility('Decidueye-hisui',16))


Loading mega from: ./home/MawileBot/src\BeatlesBoy_info.json
BST di raichu-alola a livello 100: 485
Loading team from: ./home/MawileBot/src\BeatlesBoy_info.json
[1.0, 0.16666666666666666, 0.1111111111111111, 0.0, 0, 0]
Average win percentage of team against next gym: 21.30%
Win percentage against next gym: 77.78%
Win percentage against next gym: 94.44%
5.69
Loading mega from: ./home/MawileBot/src\BeatlesBoy_info.json
BST di raichu-alola a livello 100: 485
Loading team from: ./home/MawileBot/src\BeatlesBoy_info.json
[1.0, 0.16666666666666666, 0.1111111111111111, 0.0, 0, 0]
Average win percentage of team against next gym: 21.30%
Win percentage against next gym: 77.78%
Win percentage against next gym: 94.44%
9.69


Loading mega from: ./home/MawileBot/src\BeatlesBoy_info.json
BST di decidueye-hisui a livello 100: 530
Loading team from: ./home/MawileBot/src\BeatlesBoy_info.json
[1.0, 0.16666666666666666, 0.1111111111111111, 0.0, 0, 0]
Average win percentage of team against next gym: 21.30%


In [28]:
import pypokedex
pypokedex.get(name='Torterra').base_stats


BaseStats(hp=95, attack=109, defense=105, sp_atk=75, sp_def=85, speed=56)

In [25]:
sum(pypokedex.get(name='charizard').base_stats)


534

In [29]:
import requests
r = requests.get("https://pokeapi.co/api/v2/pokemon/pikachu")
print(r.status_code)
print(r.text[:300])

200
{"id":25,"name":"pikachu","base_experience":112,"height":4,"is_default":true,"order":35,"weight":60,"abilities":[{"is_hidden":false,"slot":1,"ability":{"name":"static","url":"https://pokeapi.co/api/v2/ability/9/"}},{"is_hidden":true,"slot":3,"ability":{"name":"lightning-rod","url":"https://pokeapi.c


In [4]:
import pypokedex as poke

_orig = poke.pokemon.Pokemon._extract_sprites
@staticmethod
def _safe_extract_sprites(all_sprites):
    return _orig({k: v for k, v in all_sprites.items() if "_" in k})
poke.pokemon.Pokemon._extract_sprites = _safe_extract_sprites

try:
    poke.get(name='pikachu')
except Exception as e:
    print(type(e).__name__, "-", e)
    print(type(e.__cause__).__name__ if e.__cause__ else None)

In [5]:
import requests

r = requests.get("https://pokeapi.co/api/v2/pokemon/pikachu", timeout=15)
r.raise_for_status()
sprites = r.json()["sprites"]

def walk(d, prefix=""):
    for k, v in d.items():
        path = f"{prefix}{k}"
        if isinstance(v, dict):
            walk(v, path + ".")
        else:
            flag = "  <-- NO UNDERSCORE" if "_" not in k else ""
            print(f"{path}{flag}")

walk(sprites)

other.home.front_shiny
other.home.front_female
other.home.front_default
other.home.front_shiny_female
other.showdown.back_shiny
other.showdown.back_female
other.showdown.front_shiny
other.showdown.back_default
other.showdown.front_female
other.showdown.front_default
other.showdown.back_shiny_female
other.showdown.front_shiny_female
other.dream_world.front_female
other.dream_world.front_default
other.official-artwork.front_shiny
other.official-artwork.front_default
versions.generation-i.yellow.back_gray
versions.generation-i.yellow.front_gray
versions.generation-i.yellow.back_default
versions.generation-i.yellow.front_default
versions.generation-i.yellow.back_transparent
versions.generation-i.yellow.front_transparent
versions.generation-i.red-blue.back_gray
versions.generation-i.red-blue.front_gray
versions.generation-i.red-blue.back_default
versions.generation-i.red-blue.front_default
versions.generation-i.red-blue.back_transparent
versions.generation-i.red-blue.front_transparent
versi